# Chapter 8 — End-to-End Pipeline

**Goal:** Run the full pipeline in one go — teacher data generation → fine-tuning → merge → quantize → local inference.

This chapter ties together everything from Chapters 1-7. Every step we debugged in isolation now runs as a single automated flow. By the end, you'll have a quantized, task-specialized model ready to deploy.

**The pipeline:**
```
Task definition  →  DeepSeek generates data  →  Fine-tune SmolLM  →  Evaluate  →  Quantize  →  Deploy
```

## 8.1 Define Your Task

Change these variables to customize the pipeline for any text→label or text→text task.

In [1]:
import torch, json, os, time
import numpy as np
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset
from openai import OpenAI

# ====== CONFIGURE YOUR TASK HERE ======
TASK_NAME = "classifier"
BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
OUTPUT_DIR = f"./output-{TASK_NAME}"

# Task definition
TASK_LABELS = ["bug", "feature_request", "praise", "question"]
TASK_DESCRIPTION = """\
Classify user messages for a productivity app into:
- bug: something is broken, crashing, or not working
- feature_request: asking to ADD new functionality
- praise: compliment or positive feedback
- question: asking HOW/WHAT/WHERE/WHETHER about existing features"""

# Prompt template: how the model receives input
PROMPT_TEMPLATE = "Classify: {text}\nLabel:"

# Few-shot seed examples for the teacher
SEED_EXAMPLES = [
    {"text": "app crashes when I tap search", "label": "bug"},
    {"text": "screen goes blank after update", "label": "bug"},
    {"text": "please add dark mode", "label": "feature_request"},
    {"text": "would love bulk delete option", "label": "feature_request"},
    {"text": "this app is amazing, so intuitive", "label": "praise"},
    {"text": "love the new update, runs smooth", "label": "praise"},
    {"text": "how do I change my email address", "label": "question"},
    {"text": "does the free tier include cloud backup", "label": "question"},
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Task: {TASK_NAME}")
print(f"Labels: {TASK_LABELS}")
print(f"Device: {device}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Task: classifier
Labels: ['bug', 'feature_request', 'praise', 'question']
Device: cpu


## 8.2 Generate Training Data (Teacher)

DeepSeek generates ALL training examples using few-shot seeds.

In [ ]:
api_key = os.environ.get("DEEPSEEK_API_KEY")

if not api_key:
    raise RuntimeError("Set DEEPSEEK_API_KEY to run the pipeline")

client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

# Build few-shot system prompt
seed_text = "\n".join(
    f'  {{"text": "{ex["text"]}", "label": "{ex["label"]}"}}'
    for ex in SEED_EXAMPLES
)

system_prompt = f"""\
{TASK_DESCRIPTION}

Examples of correct labeling:
{seed_text}

Return ONLY a JSON array of {{"text": "...", "label": "..."}} objects. No markdown, no explanation."""

# Generate data
teacher_data = []
BATCHES = 40

print(f"Generating {BATCHES} batches from DeepSeek...")
for i in range(BATCHES):
    raw = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "Generate 10 new labeled examples. Vary phrasing and vocabulary. Include all categories."},
        ],
        temperature=0.9, max_tokens=1024,
    ).choices[0].message.content

    try:
        batch = json.loads(raw)
        for item in batch:
            text = item.get("text", "").strip()
            label = item.get("label", "").strip().lower()
            if text and label in set(TASK_LABELS):
                teacher_data.append({"text": text, "label": label})
        if (i + 1) % 10 == 0:
            print(f"  Batch {i+1}/{BATCHES}: {len(teacher_data)} examples  {dict(Counter(item['label'] for item in teacher_data))}")
    except json.JSONDecodeError:
        pass
    time.sleep(0.3)

print(f"\nGenerated {len(teacher_data)} teacher-labeled examples")
print(f"Class distribution: {dict(Counter(item['label'] for item in teacher_data))}")

Generating 40 batches from DeepSeek...
  Batch 10/40: 90 examples  {'bug': 27, 'feature_request': 27, 'praise': 18, 'question': 18}
  Batch 20/40: 170 examples  {'bug': 51, 'feature_request': 51, 'praise': 34, 'question': 34}
  Batch 30/40: 270 examples  {'bug': 81, 'feature_request': 80, 'praise': 54, 'question': 55}
  Batch 40/40: 370 examples  {'bug': 110, 'feature_request': 110, 'praise': 75, 'question': 75}

Generated 370 teacher-labeled examples
Class distribution: {'bug': 110, 'feature_request': 110, 'praise': 75, 'question': 75}


## 8.3 Format & Tokenize

Convert to prompt→completion pairs, tokenize with masked labels.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Format as prompt→completion
raw_data = []
for item in teacher_data:
    prompt = PROMPT_TEMPLATE.format(text=item["text"])
    completion = f" {item['label']}"
    raw_data.append({"prompt": prompt, "completion": completion})

np.random.seed(42)
np.random.shuffle(raw_data)
split = int(0.85 * len(raw_data))
train_data = raw_data[:split]
eval_data = raw_data[split:]
print(f"Train: {len(train_data)}  |  Eval: {len(eval_data)}")

# Tokenize
def tokenize(ex):
    p = tokenizer(ex["prompt"], truncation=True, max_length=128, padding=False)
    c = tokenizer(ex["completion"], truncation=True, max_length=32, padding=False)
    input_ids = p["input_ids"] + c["input_ids"] + [tokenizer.eos_token_id]
    labels = [-100] * len(p["input_ids"]) + c["input_ids"] + [tokenizer.eos_token_id]
    return {"input_ids": input_ids, "attention_mask": [1]*len(input_ids), "labels": labels}

train_dataset = Dataset.from_list(train_data).map(tokenize, remove_columns=["prompt", "completion"])
eval_dataset = Dataset.from_list(eval_data).map(tokenize, remove_columns=["prompt", "completion"])
print(f"Tokenized: {len(train_dataset)} train / {len(eval_dataset)} eval")

Train: 314  |  Eval: 56


Map: 100%|██████████| 56/56 [00:00<00:00, 1296.47 examples/s]

Tokenized: 314 train / 56 eval


## 8.4 Fine-Tune with LoRA

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id]*pad)
        attention_mask.append(x["attention_mask"] + [0]*pad)
        labels.append(x["labels"] + [-100]*pad)
    return {"input_ids": torch.tensor(input_ids), "attention_mask": torch.tensor(attention_mask), "labels": torch.tensor(labels)}

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoint",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=(device == "cuda"),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=collate_fn)
print("Training...")
trainer.train()

# Save adapter
model.save_pretrained(f"{OUTPUT_DIR}/adapter")
print(f"Adapter saved to {OUTPUT_DIR}/adapter")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1108.38it/s]


Trainable: 460,800 / 134,975,808 (0.3%)
Training...


C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.729078,3.059416
2,0.785093,0.407361
3,0.081568,0.067954
4,0.057977,0.033691
5,0.024904,0.026301
6,0.036009,0.019104
7,0.017024,0.019573
8,0.013177,0.016729
9,0.006008,0.016885
10,0.017098,0.017368


C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\guhar\AppDa

Adapter saved to ./output-classifier/adapter


## 8.5 Evaluate the Fine-Tuned Model

In [6]:
def predict(text):
    prompt = PROMPT_TEMPLATE.format(text=text)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract after prompt marker
    marker = PROMPT_TEMPLATE.split("{")[0].strip().split(":")[-1].strip() + ":"
    if marker in full:
        raw = full.split(marker)[-1].strip()
        for label in TASK_LABELS:
            if raw.startswith(label):
                return label
    return raw.split()[0] if raw.split() else "???"

# Quick test
test_cases = [
    "the app keeps crashing when I upload a photo",
    "please add dark mode support",
    "love the new dashboard design",
    "how do I export my data",
]
print("Quick test:\n")
for text in test_cases:
    print(f"  {text[:60]}...")
    print(f"  → {predict(text)}\n")

Quick test:

  the app keeps crashing when I upload a photo...
  → bug

  please add dark mode support...
  → feature_request

  love the new dashboard design...
  → praise

  how do I export my data...
  → question



## 8.6 Merge Adapter → Quantize → Deploy

Bake the LoRA adapter into base weights, then quantize to INT4 for deployment.

In [7]:
# Merge adapter into base model
print("Merging adapter...")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
merged = PeftModel.from_pretrained(base, f"{OUTPUT_DIR}/adapter")
merged = merged.merge_and_unload()

merged_path = f"{OUTPUT_DIR}/merged"
merged.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

def get_size(path):
    total = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, files in os.walk(path) for f in files)
    return total / (1024*1024)

print(f"Merged FP32: {get_size(merged_path):.0f} MB")

# Quantize to INT4
print("Quantizing to INT4...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
)
deploy_model = AutoModelForCausalLM.from_pretrained(
    merged_path, quantization_config=quant_config,
    device_map="auto" if device == "cuda" else None,
)
deploy_model.eval()
print(f"INT4 model ready — ~{get_size(merged_path)/8:.0f} MB equivalent")

# Test quantized inference
print("\nQuantized model predictions:\n")
for text in test_cases:
    prompt = PROMPT_TEMPLATE.format(text=text)
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = deploy_model.generate(**inputs, max_new_tokens=5, do_sample=False,
                                    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    label = "???"
    if "Label:" in full:
        raw = full.split("Label:")[-1].strip()
        for l in TASK_LABELS:
            if raw.startswith(l):
                label = l
                break
    print(f"  {text[:60]}... → {label}")

Merging adapter...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]


Merged FP32: 517 MB
Quantizing to INT4...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 272/272 [00:04<00:00, 54.55it/s]
C:\Users\guhar\AppData\Roaming\Python\Python312\site-packages\bitsandbytes\backends\cpu\ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
C:\Users\guhar\AppData\Roaming\Python\Python312\site-pac

INT4 model ready — ~65 MB equivalent

Quantized model predictions:

  the app keeps crashing when I upload a photo... → bug
  please add dark mode support... → feature_request
  love the new dashboard design... → praise
  how do I export my data... → question


## 8.7 Pipeline Summary

| Step | What happened | Output |
|---|---|---|
| 8.1 | Defined task + few-shot seeds | Configuration |
| 8.2 | DeepSeek generated ~400 labeled examples | Training data |
| 8.3 | Formatted prompt→completion pairs, tokenized | Tokenized dataset |
| 8.4 | Fine-tuned SmolLM with LoRA (460K params trained) | LoRA adapter (~2MB) |
| 8.5 | Evaluated on test inputs | Accuracy check |
| 8.6 | Merged adapter → quantized to INT4 | Deployable model (~68MB) |

**What we built:**
- A 135M-parameter model specialized for one task
- Trained entirely on teacher-generated data (pure distillation)
- Compressed to ~68MB — runs on laptop CPU, any smartphone, Raspberry Pi
- No GPU needed for inference

**To deploy further:** convert the merged model to GGUF format (see Chapter 7.6) and run with llama.cpp or Ollama.

**To customize for a new task:** change the variables in Section 8.1 and re-run. The pipeline is task-agnostic — it works for any text→text task you can define with labels and few-shot examples.